In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def extract_chess_squares(image_path):
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"Could not load image at {image_path}")

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    height, width, _ = img.shape


    sq_height = height // 8
    sq_width = width // 8

    squares = []

    for y in range(8):
        for x in range(8):

            start_y = y * sq_height
            end_y = start_y + sq_height

            start_x = x * sq_width
            end_x = start_x + sq_width

            square_crop = img[start_y:end_y, start_x:end_x]
            squares.append(square_crop)

    return squares

image_file = "pieces_sample_1.jpg"
try:
    board_squares = extract_chess_squares(image_file)
    print(f"Successfully extracted {len(board_squares)} squares.")

    # first 4 squares (top Left corner)
    fig, axes = plt.subplots(1, 16, figsize=(18, 3))
    for i in range(16):
        axes[i].imshow(board_squares[i])
        axes[i].axis('off')
        axes[i].set_title(f"Square {i}")
    plt.show()

except Exception as e:
    print(e)

In [ ]:
import cv2
import os
import glob

def process_and_slice_directory(input_dir, output_base_dir, category_prefix):
    """
    Reads all JPGs in input_dir, slices them into an 8x8 grid,
    and saves them to output_base_dir/sampleX/.
    """
    os.makedirs(output_base_dir, exist_ok=True)
    image_paths = glob.glob(os.path.join(input_dir, "*.jpg"))

    if not image_paths:
        print(f"No JPGs found in {input_dir}. Check your paths!")
        return

    for image_path in image_paths:

        filename = os.path.basename(image_path)
        # We split by '_' to get "1.jpg", then split by '.' to isolate "1"
        sample_num = filename.split('_')[-1].split('.')[0]

        sample_out_dir = os.path.join(output_base_dir, f"sample{sample_num}")
        os.makedirs(sample_out_dir, exist_ok=True)

        img = cv2.imread(image_path)
        if img is None:
            print(f"Failed to read {image_path}. Skipping.")
            continue

        height, width, _ = img.shape
        sq_height = height // 8
        sq_width = width // 8

        for y in range(8):
            for x in range(8):

                start_y = y * sq_height
                end_y = start_y + sq_height
                start_x = x * sq_width
                end_x = start_x + sq_width

                square_crop = img[start_y:end_y, start_x:end_x]

                square_filename = f"{category_prefix}_{x}{y}.jpg"
                save_path = os.path.join(sample_out_dir, square_filename)

                cv2.imwrite(save_path, square_crop)

        print(f"Successfully processed {filename} -> Saved to {sample_out_dir}/")

# #empty squares
# empty_input_path = "empty_samples"
# empty_output_path = os.path.join("extracted_squares", "extracted_empty")
# process_and_slice_directory(empty_input_path, empty_output_path, "empty")

#pieces squares
pieces_input_path = "data/extracted_boards"
pieces_output_path = os.path.join("data/extracted_squares", "extracted_pieces")
process_and_slice_directory(pieces_input_path, pieces_output_path, "pieces")

print("\nAll slicing completed successfully!")